# Notebook Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

# Import Project Modules

In [ ]:
from src.dataset import load_data, make_data_loader, make_windows
from src.model import StockMLP, StockCNN
from src.train import train
from src.evaluate import evaluate

# Configuration

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
BATCH_SIZE = 128
WINDOW_SIZE = 30
EPOCHS = 15
USE_CNN = False  # switch between MLP and CNN
NORMALIZE = True

# Load Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please place train.csv inside data/ directory."
    )

df = load_data(DATA_PATH)
df.head()

# Creating Sliding Windows

In [ ]:
X, y = make_windows(df, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive ratio:", y.mean())


# Train / Validation Split

In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))

# DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

if NORMALIZE:
    from src.features import normalize
    X_train = normalize(X_train)
    X_val = normalize(X_val)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize Model

In [ ]:
if USE_CNN:
    model = StockCNN()
    print("Using CNN model")
else:
    model = StockMLP(input_dim=WINDOW_SIZE)
    print("Using MLP model")

model

# Train Model

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

train(
    model=model,
    loader=train_loader,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
)

# Evaluate on Validation Set

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

val_accuracy = evaluate(
    model,
    X_val,
    y_val,
    device=device
)

print(f"Validation Accuracy: {val_accuracy:.4f}")

# Save Trained Model

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "stock_model_v1.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

# Quick Sanity Prediction

In [ ]:
model.eval()

sample = torch.tensor(X_val[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(sample)
    probs = torch.sigmoid(logits)

print("Predicted probabilities:", probs.cpu().numpy())
print("True labels:", y_val[:5])